# Vietnamese News RAG: Colab + Cloudflare demo

Chọn **Runtime > Change runtime type > T4 GPU**. Upload `rag_colab_data.zip` vào `MyDrive/`, rồi chạy tuần tự. Cell cuối phải tiếp tục chạy để API còn online.

In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = 'https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News.git'
REPO_REF = 'test_feature_ta'  # Đổi nếu đã merge sang branch khác.
REPO_DIR = Path('/content/Text-Mining---RAG-on-News')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements/backend.txt'], check=True)

In [ ]:
from google.colab import drive
import shutil, zipfile

drive.mount('/content/drive')
bundle = Path('/content/drive/MyDrive/rag_colab_data.zip')
if not bundle.is_file():
    raise FileNotFoundError(f'Upload bundle trước: {bundle}')
data_dir = REPO_DIR / 'data'
if data_dir.exists():
    shutil.rmtree(data_dir)
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(REPO_DIR)
assert (data_dir / 'qdrant_news').is_dir()
assert (data_dir / 'qdrant_news_bm25.pkl').is_file()
print('RAG index restored')

In [ ]:
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN') or ''
except Exception:
    hf_token = ''
os.environ.update({
    'QDRANT_PATH': 'data/qdrant_news',
    'BM25_INDEX_PATH': 'data/qdrant_news_bm25.pkl',
    'QDRANT_COLLECTION': 'news_bge_token',
    'MODEL_DEVICE': 'cuda:0',
    'MODEL_DTYPE': 'float16',
    'LLM_PROVIDER': 'hf_model',
    'HF_LLM_MODEL': 'CohereLabs/aya-expanse-8b',
    'HF_LLM_DEVICE': 'cuda:0',
    'HF_LLM_LOAD_IN_4BIT': 'true',
    'HF_LLM_MAX_NEW_TOKENS': '512',
    'CORS_ORIGINS': '*',
    'HF_TOKEN': hf_token,
})
import torch
assert torch.cuda.is_available(), 'Hãy bật T4 GPU cho runtime'
print(torch.cuda.get_device_name(0))

In [ ]:
# Cell này giữ backend và tunnel sống. Dừng cell sẽ tắt URL công khai.
!python -m deployment.run_colab_demo

Khi thấy `PUBLIC_API_URL`, kiểm tra `<URL>/api/health`. Frontend dùng `VITE_API_BASE_URL=<URL>`, sau đó build/deploy lại frontend.